# BS/Gridworld LLM State Workshop

This notebook is intentionally linear and explicit:
- no user-defined helper functions,
- direct state/message inspection,
- direct `llm.chat(...)` calls,
- manual/action override support.

Use this to test a new model family against exact prompts from `BSEnvironment` or `GridWorldEnvironment`.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys
import json
import re
from pprint import pprint

from vllm import LLM, SamplingParams


In [ ]:
ENV_NAME = "gridworld"  # "bs" or "gridworld"

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
TRUST_REMOTE_CODE = True
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 4096
MODEL_SEED = 0

TEMPERATURE = 0.6
TOP_P = 0.95
MAX_TOKENS = 512
REPETITION_PENALTY = 1.0
GENERATION_SEED = 0

N_RESPONSES_SAME_STATE = 4
ROLLOUT_STEPS = 6

# BS env knobs
BS_NUM_PLAYERS = 3
BS_CARDS_PER_PLAYER = 5

# Gridworld env knobs
GRID_WIDTH = 9
GRID_HEIGHT = 9
GRID_WALL_PROB = 0.18
GRID_MAX_TRIES = 200
GRID_MAX_STEPS = 30
GRID_VIEW_RADIUS = 2
GRID_AUTO_MOVE_EXPLORER = True
GRID_FIXED = None

# GRID_FIXED example:
# GRID_FIXED = [
#     "#########",
#     "#S..#...#",
#     "#.#.#.#.#",
#     "#.#...#.#",
#     "#.###.#.#",
#     "#...#...#",
#     "#.#.###.#",
#     "#...#..G#",
#     "#########",
# ]


In [ ]:
repo_root = Path('/playpen-ssd/smerrill/deception2')
bs_src = repo_root / 'BS' / 'src'
grid_src = repo_root / 'Gridworld' / 'src'
core_src = repo_root / 'src'

for p in [repo_root, bs_src, grid_src, core_src]:
    if str(p) not in sys.path:
        sys.path.append(str(p))

print('repo_root:', repo_root)
print('ENV_NAME:', ENV_NAME)
print('Added to sys.path:', bs_src, grid_src, core_src)


In [ ]:
if ENV_NAME.lower() == 'bs':
    from bs_environment import BSEnvironment
    from deck import Deck

    names = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank']
    agents = [SimpleNamespace(name=names[i]) for i in range(BS_NUM_PLAYERS)]
    env = BSEnvironment(agents=agents, seed=MODEL_SEED)

    if BS_CARDS_PER_PLAYER != 5:
        env.deck = Deck(seed=MODEL_SEED)
        env.deck.shuffle()
        env.deal(n_cards=BS_CARDS_PER_PLAYER)

elif ENV_NAME.lower() == 'gridworld':
    from gridworld_environment import GridWorldEnvironment, GridSpec

    guide = SimpleNamespace(name='Guide')
    explorer = SimpleNamespace(name='Explorer')

    grid_spec = GridSpec(
        width=GRID_WIDTH,
        height=GRID_HEIGHT,
        wall_prob=GRID_WALL_PROB,
        max_tries=GRID_MAX_TRIES,
    )

    env = GridWorldEnvironment(
        agents=[guide, explorer],
        seed=MODEL_SEED,
        grid_spec=grid_spec,
        grid=GRID_FIXED,
        max_steps=GRID_MAX_STEPS,
        view_radius=GRID_VIEW_RADIUS,
        auto_move_explorer=GRID_AUTO_MOVE_EXPLORER,
    )

else:
    raise ValueError("ENV_NAME must be 'bs' or 'gridworld'.")

print('Environment built.')
print('phase:', env.phase)
print('active_player_idx:', env.active_player_idx)
print('active_player:', None if env.active_player_idx is None else env.agents[env.active_player_idx].name)


In [ ]:
state = env.get_state(include_system_prompt=True)

print('=== STATE SUMMARY ===')
for k in ['phase', 'active_player', 't', 'max_steps', 'pos', 'done', 'outcome']:
    if k in state:
        print(f'{k}: {state[k]}')

if ENV_NAME.lower() == 'bs':
    print('
=== BS TABLE ===')
    print('current_rank:', env.current_rank)
    print('pile_size:', len(env.pile))
    for p in env.agents:
        print(f'{p.name}: {len(p.hand)} cards | hand={p.hand}')

if ENV_NAME.lower() == 'gridworld':
    print('
=== GRID (full) ===')
    print(env.render(full=True))
    print('
=== GRID (local) ===')
    print(env.render(full=False, radius=env.view_radius))
    print('
optimal_moves:', env.optimal_moves(env.pos))
    print('last_recommendation:', env.last_recommendation)

print('
=== EXACT MESSAGES TO MODEL ===')
for i, m in enumerate(state['messages']):
    print(f"
--- message[{i}] role={m.get('role')} ---")
    print(m.get('content', ''))


In [ ]:
llm = LLM(
    model=MODEL_NAME,
    seed=MODEL_SEED,
    trust_remote_code=TRUST_REMOTE_CODE,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
)

tokenizer = llm.get_tokenizer()
print('Loaded model:', MODEL_NAME)
print('Tokenizer:', getattr(tokenizer, 'name_or_path', 'unknown'))


In [ ]:
messages = state['messages']
prompt_text = None

try:
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    print(prompt_text)
except Exception as e:
    print('chat_template render failed:', e)


In [ ]:
messages = state['messages']

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    repetition_penalty=REPETITION_PENALTY,
    seed=GENERATION_SEED,
)

outputs = llm.chat(messages=[messages], sampling_params=[sampling_params])
raw_text = outputs[0].outputs[0].text

print('=== RAW MODEL OUTPUT ===')
print(raw_text)

json_match = re.search(r"\{[\s\S]*?\}", raw_text)
parsed_action = None

if json_match is None:
    print('
No JSON object found in raw output.')
else:
    json_text = json_match.group(0)
    json_text = re.sub(r"//.*?$", "", json_text, flags=re.MULTILINE)
    json_text = re.sub(r"#.*?$", "", json_text, flags=re.MULTILINE)
    json_text = re.sub(r",\s*([\]}])", r"\1", json_text)

    try:
        parsed_action = json.loads(json_text)
        print('
=== PARSED ACTION ===')
        pprint(parsed_action, sort_dicts=False)
    except Exception as e:
        print('
JSON parse failed:', e)
        print('candidate JSON text:')
        print(json_text)


In [ ]:
action_to_apply = parsed_action

if not isinstance(action_to_apply, dict):
    action_to_apply = {}

if ENV_NAME.lower() == 'bs':
    if state['phase'] == 'PLAY':
        action_to_apply['Action'] = 'PLAY'
        if 'Cards_played' not in action_to_apply or action_to_apply['Cards_played'] is None:
            action_to_apply['Cards_played'] = []
        if isinstance(action_to_apply['Cards_played'], str):
            action_to_apply['Cards_played'] = [action_to_apply['Cards_played']]

    elif state['phase'] == 'CHALLENGE':
        act = str(action_to_apply.get('Action', 'Pass')).strip().capitalize()
        if act not in {'Challenge', 'Pass'}:
            act = 'Pass'
        action_to_apply = {'Action': act}

if ENV_NAME.lower() == 'gridworld':
    if state['phase'] == 'GUIDE':
        direction = action_to_apply.get('Direction', action_to_apply.get('direction', 'UP'))
        direction = str(direction).strip().upper()

        if direction == 'U':
            direction = 'UP'
        if direction == 'D':
            direction = 'DOWN'
        if direction == 'L':
            direction = 'LEFT'
        if direction == 'R':
            direction = 'RIGHT'
        if direction not in {'UP', 'DOWN', 'LEFT', 'RIGHT'}:
            direction = 'UP'

        action_to_apply = {
            'Action': 'RECOMMEND',
            'Direction': direction,
            'Message': str(action_to_apply.get('Message', '')),
        }

    elif state['phase'] == 'CHALLENGE':
        act = str(action_to_apply.get('Action', 'Pass')).strip().capitalize()
        if act not in {'Challenge', 'Pass'}:
            act = 'Pass'
        action_to_apply = {'Action': act}

    elif state['phase'] == 'MOVE':
        action_to_apply = {'Action': 'AUTO'}

print('=== ACTION TO APPLY ===')
pprint(action_to_apply, sort_dicts=False)

step_out = env.manual_step(action_to_apply, debug=True)
print('
=== STEP RESULT ===')
pprint(step_out, sort_dicts=False)


In [ ]:
state = env.get_state(include_system_prompt=True)

print('=== NEXT STATE SUMMARY ===')
for k in ['phase', 'active_player', 't', 'max_steps', 'pos', 'done', 'outcome']:
    if k in state:
        print(f'{k}: {state[k]}')

if ENV_NAME.lower() == 'bs':
    print('
=== BS TABLE ===')
    print('current_rank:', env.current_rank)
    print('pile_size:', len(env.pile))
    for p in env.agents:
        print(f'{p.name}: {len(p.hand)} cards | hand={p.hand}')

if ENV_NAME.lower() == 'gridworld':
    print('
=== GRID (full) ===')
    print(env.render(full=True))
    print('
=== GRID (local) ===')
    print(env.render(full=False, radius=env.view_radius))
    print('
optimal_moves:', env.optimal_moves(env.pos))
    print('last_recommendation:', env.last_recommendation)

print('
=== NEXT MESSAGES TO MODEL ===')
for i, m in enumerate(state['messages']):
    print(f"
--- message[{i}] role={m.get('role')} ---")
    print(m.get('content', ''))


In [ ]:
MANUAL_ACTION = None

# BS PLAY example:
# MANUAL_ACTION = {"Action": "PLAY", "Cards_played": ["2H"]}

# BS CHALLENGE example:
# MANUAL_ACTION = {"Action": "Challenge"}

# Gridworld GUIDE example:
# MANUAL_ACTION = {"Action": "RECOMMEND", "Direction": "LEFT", "Message": ""}

# Gridworld CHALLENGE example:
# MANUAL_ACTION = {"Action": "Pass"}

# Gridworld MOVE example (only needed when auto_move_explorer=False):
# MANUAL_ACTION = {"Action": "AUTO"}

if MANUAL_ACTION is None:
    print('Set MANUAL_ACTION and re-run this cell.')
else:
    override_result = env.manual_step(MANUAL_ACTION, debug=True)
    print('override_result:')
    pprint(override_result, sort_dicts=False)

    state = env.get_state(include_system_prompt=True)
    print('
new phase:', state['phase'], '| active_player:', state['active_player'])


In [ ]:
snapshot = env._snapshot_state()
state_probe = env.get_state(include_system_prompt=True)
messages_probe = state_probe['messages']

params_list = []
for i in range(N_RESPONSES_SAME_STATE):
    params_list.append(
        SamplingParams(
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            repetition_penalty=REPETITION_PENALTY,
            seed=GENERATION_SEED + i,
        )
    )

batch_messages = [messages_probe for _ in range(N_RESPONSES_SAME_STATE)]
batch_outputs = llm.chat(messages=batch_messages, sampling_params=params_list)

print('phase:', state_probe['phase'], '| active_player:', state_probe['active_player'])

for i, out in enumerate(batch_outputs):
    raw = out.outputs[0].text
    print(f"
===== SAMPLE {i} | seed={GENERATION_SEED + i} =====")
    print(raw)

    m = re.search(r"\{[\s\S]*?\}", raw)
    if m is None:
        print('No JSON found.')
        continue

    candidate = m.group(0)
    candidate = re.sub(r"//.*?$", "", candidate, flags=re.MULTILINE)
    candidate = re.sub(r"#.*?$", "", candidate, flags=re.MULTILINE)
    candidate = re.sub(r",\s*([\]}])", r"\1", candidate)

    parsed = None
    try:
        parsed = json.loads(candidate)
        print('Parsed:', parsed)
    except Exception as e:
        print('Parse error:', e)
        print('Candidate:', candidate)
        continue

    env._restore_state(snapshot)

    action_test = parsed
    if not isinstance(action_test, dict):
        action_test = {}

    if ENV_NAME.lower() == 'bs':
        if state_probe['phase'] == 'PLAY':
            action_test['Action'] = 'PLAY'
            if 'Cards_played' not in action_test or action_test['Cards_played'] is None:
                action_test['Cards_played'] = []
            if isinstance(action_test['Cards_played'], str):
                action_test['Cards_played'] = [action_test['Cards_played']]
        elif state_probe['phase'] == 'CHALLENGE':
            act = str(action_test.get('Action', 'Pass')).strip().capitalize()
            if act not in {'Challenge', 'Pass'}:
                act = 'Pass'
            action_test = {'Action': act}

    if ENV_NAME.lower() == 'gridworld':
        if state_probe['phase'] == 'GUIDE':
            d = action_test.get('Direction', action_test.get('direction', 'UP'))
            d = str(d).strip().upper()
            if d == 'U':
                d = 'UP'
            if d == 'D':
                d = 'DOWN'
            if d == 'L':
                d = 'LEFT'
            if d == 'R':
                d = 'RIGHT'
            if d not in {'UP', 'DOWN', 'LEFT', 'RIGHT'}:
                d = 'UP'
            action_test = {'Action': 'RECOMMEND', 'Direction': d, 'Message': str(action_test.get('Message', ''))}
        elif state_probe['phase'] == 'CHALLENGE':
            act = str(action_test.get('Action', 'Pass')).strip().capitalize()
            if act not in {'Challenge', 'Pass'}:
                act = 'Pass'
            action_test = {'Action': act}
        elif state_probe['phase'] == 'MOVE':
            action_test = {'Action': 'AUTO'}

    step_test = env.manual_step(action_test, debug=False)
    next_state = env.get_state(include_system_prompt=False)

    print('Applied:', action_test)
    print('step_result.phase:', step_test.get('phase'), '| next_phase:', next_state.get('phase'), '| active_player:', next_state.get('active_player'))

    if ENV_NAME.lower() == 'gridworld':
        print('pos:', next_state.get('pos'), '| done:', next_state.get('done'), '| outcome:', next_state.get('outcome'))

env._restore_state(snapshot)
print('
Restored original environment state.')


In [ ]:
for step_idx in range(ROLLOUT_STEPS):
    state = env.get_state(include_system_prompt=True)

    print(f"
================ STEP {step_idx} ================")
    print('phase:', state['phase'], '| active_player:', state['active_player'])

    if state.get('done'):
        print('Environment already done:', state.get('outcome'))
        break

    params = SamplingParams(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_TOKENS,
        repetition_penalty=REPETITION_PENALTY,
        seed=GENERATION_SEED + 1000 + step_idx,
    )

    out = llm.chat(messages=[state['messages']], sampling_params=[params])[0]
    raw_text = out.outputs[0].text

    print('
RAW OUTPUT:')
    print(raw_text)

    m = re.search(r"\{[\s\S]*?\}", raw_text)
    parsed = {}

    if m is not None:
        candidate = m.group(0)
        candidate = re.sub(r"//.*?$", "", candidate, flags=re.MULTILINE)
        candidate = re.sub(r"#.*?$", "", candidate, flags=re.MULTILINE)
        candidate = re.sub(r",\s*([\]}])", r"\1", candidate)
        try:
            parsed = json.loads(candidate)
            print('PARSED:', parsed)
        except Exception as e:
            print('PARSE ERROR:', e)
            parsed = {}
    else:
        print('No JSON found.')

    action_to_apply = parsed
    if not isinstance(action_to_apply, dict):
        action_to_apply = {}

    if ENV_NAME.lower() == 'bs':
        if state['phase'] == 'PLAY':
            action_to_apply['Action'] = 'PLAY'
            if 'Cards_played' not in action_to_apply or action_to_apply['Cards_played'] is None:
                action_to_apply['Cards_played'] = []
            if isinstance(action_to_apply['Cards_played'], str):
                action_to_apply['Cards_played'] = [action_to_apply['Cards_played']]
        elif state['phase'] == 'CHALLENGE':
            act = str(action_to_apply.get('Action', 'Pass')).strip().capitalize()
            if act not in {'Challenge', 'Pass'}:
                act = 'Pass'
            action_to_apply = {'Action': act}

    if ENV_NAME.lower() == 'gridworld':
        if state['phase'] == 'GUIDE':
            d = action_to_apply.get('Direction', action_to_apply.get('direction', 'UP'))
            d = str(d).strip().upper()
            if d == 'U':
                d = 'UP'
            if d == 'D':
                d = 'DOWN'
            if d == 'L':
                d = 'LEFT'
            if d == 'R':
                d = 'RIGHT'
            if d not in {'UP', 'DOWN', 'LEFT', 'RIGHT'}:
                d = 'UP'
            action_to_apply = {'Action': 'RECOMMEND', 'Direction': d, 'Message': str(action_to_apply.get('Message', ''))}
        elif state['phase'] == 'CHALLENGE':
            act = str(action_to_apply.get('Action', 'Pass')).strip().capitalize()
            if act not in {'Challenge', 'Pass'}:
                act = 'Pass'
            action_to_apply = {'Action': act}
        elif state['phase'] == 'MOVE':
            action_to_apply = {'Action': 'AUTO'}

    print('APPLY:', action_to_apply)
    step_out = env.manual_step(action_to_apply, debug=True)
    print('STEP_OUT:', step_out)

    if ENV_NAME.lower() == 'gridworld':
        print('POS:', env.pos, '| t:', env.t, '| done:', env.done, '| outcome:', env.outcome)

    if ENV_NAME.lower() == 'bs':
        print('current_rank:', env.current_rank, '| pile_size:', len(env.pile))
        hand_counts = {a.name: len(a.hand) for a in env.agents}
        print('hand_counts:', hand_counts)
